# 05 · Portfolio backtest

**Goal:** turn the signal into portfolios and test them honestly — next-open execution, transaction costs, overlapping holding periods, and a scoreboard across strategies and holding days. No alpha claim survives unless it clears the net-of-cost gate **and** the out-of-sample / robustness checks.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / 'src'))
import pandas as pd, numpy as np
from reddit_hype.config import load_settings, credential_report
from reddit_hype.utils import read_parquet_or_empty
s = load_settings()
print(credential_report())
from reddit_hype.backtester import run_strategy_grid
from reddit_hype import reporting as rp
panel = read_parquet_or_empty(s.path('panel'))
prices = read_parquet_or_empty(s.path('prices'))

In [ ]:
scoreboard, results = run_strategy_grid(panel, prices, s)
scoreboard[['strategy','holding_days','sharpe','ann_return','max_drawdown','hit_rate','turnover','research_only']].head(15)

### Equity curves (net of costs)

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(10,5))
for k,r in sorted(results.items(), key=lambda kv:-kv[1].stats.get('sharpe',-9))[:6]:
    if not r.daily_returns.empty:
        (1+r.daily_returns.fillna(0)).cumprod().plot(ax=ax, label=k)
ax.legend(fontsize=7); ax.set_title('Top strategies — growth of $1'); None

**Reminder:** on synthetic/mock data these curves are random by construction — the point is the *machinery*, not the numbers. Re-run on real data before concluding.